# easySLM - Minimal Fine-Tuning & Inference Quickstart

This notebook demonstrates the minimal Python API workflow:
1. Fine-tune `Qwen/Qwen2.5-0.5B` on `data/agnews_tiny` using `FineTuningConfig` built-in defaults (checkpoints saved to `~/.easyslm/finetuned`).
2. Inspect per-class validation metrics (Precision, Recall, F1) from the saved evaluation artifacts.
3. Reload the best checkpoint via `SLMBase.from_best_model(...)` and run `predict_texts(...)` on custom inputs.

In [ ]:
from src import FineTuningConfig, SLMBase, SLMFineTuner
from src.utils import get_project_root

# 1. Define training configuration:
config = FineTuningConfig(
    mode="cls",
    num_train_epochs=2.0,
    use_mlflow=False,
)

# 2. Instantiate the fine-tuner and train on data/agnews_tiny (auto-discovers train.txt & val.txt)
tuner = SLMFineTuner(model_name="Qwen/Qwen2.5-0.5B", config=config)
results = tuner.train(train_data_path=get_project_root() / "data" / "agnews_tiny")

print("Saved checkpoint:", results["checkpoint_dir"])

In [ ]:
import json
from pathlib import Path

# 3. Display per-class validation metrics (Precision, Recall, F1) from evaluation artifacts
metrics_file = Path(results["checkpoint_dir"]) / "tables" / "per_class_metrics.json"
per_class = json.loads(metrics_file.read_text(encoding="utf-8"))

print(f"{'Class':<15} | {'Precision':>9} | {'Recall':>9} | {'F1':>9}")
print("-" * 51)
for cls_name, prec, rec, f1 in zip(
    per_class["class_name"],
    per_class["precision"],
    per_class["recall"],
    per_class["f1_score"],
    strict=False,
):
    print(f"{cls_name:<15} | {prec:>9.4f} | {rec:>9.4f} | {f1:>9.4f}")

In [ ]:
# 4. Automatically load the best checkpoint using Model Cards
model = SLMBase.from_best_model(
    model_type="cls",
    metric="eval_f1_macro",
)

# 5. Predict on 3 custom news headlines
custom_texts = [
    "Tech giant unveils next-generation quantum processor for cloud AI workloads.",
    "Central bank holds interest rates steady as quarterly corporate earnings beat forecasts.",
    "National football squad secures dramatic overtime victory in the championship final.",
]

predictions = model.predict_texts(custom_texts)
for item in predictions:
    print(f"Text:        {item['text']}")
    print(f"Prediction:  {item['predictions']}")
    print(f"Confidence:  {item.get('probabilities', item.get('raw_output'))}\n")